<a href="https://colab.research.google.com/github/74santos/Professional-UX-Auditor/blob/main/UX_Site_Auditor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import requests
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO
from collections import Counter
import pandas as pd
import time

# --- PART 1: ACCESSIBILITY AUDITOR ---


class DigitalAuditor:
    def __init__(self, url):
        self.url = url.rstrip('/')
        self.headers = {'User-Agent': 'Mozilla/5.0'}
        self.results = {"Accessibility": [], "Links": [], "Palette": []}

    def run_full_audit(self):
        print(f"🚀 Initializing Master Audit for: {self.url}\n" + "-"*30)
        try:
            response = requests.get(self.url, headers=self.headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            # 1. ACCESSIBILITY & COLOR EXTRACTION
            images = soup.find_all('img')
            for img in images:
                src = img.get('src', '')
                alt = img.get('alt', '')
                status = "✅ Pass" if alt.strip() else "❌ FAIL: Missing Alt Text"
                self.results["Accessibility"].append({"Source": src[:50], "Status": status})

            # Extract palette from the first valid image (usually the logo)
            if images:
                logo_url = images[0].get('src')
                if not logo_url.startswith('http'):
                    logo_url = self.url + (logo_url if logo_url.startswith('/') else '/' + logo_url)
                self.results["Palette"] = self.extract_colors(logo_url)

            # 2. LINK INTEGRITY (Top 10 for speed)
            links = soup.find_all('a', href=True)
            for link in links[:10]:
                href = link['href']
                if href.startswith('/'): href = self.url + href
                if not href.startswith('http'): continue

                try:
                    res = requests.head(href, headers=self.headers, timeout=5)
                    health = "✅ OK" if res.status_code == 200 else f"❌ Issue ({res.status_code})"
                except:
                    health = "❌ Connection Failed"

                self.results["Links"].append({"URL": href, "Health": health})
                time.sleep(0.3) # Polite scraping

        except Exception as e:
            print(f"CRITICAL ERROR: {e}")

    def extract_colors(self, img_url):
        try:
            res = requests.get(img_url, timeout=5)
            img = Image.open(BytesIO(res.content)).convert('RGB').resize((50, 50))
            # FIX: Properly unpack the RGB tuple from the Counter object
            most_common = Counter(list(img.getdata())).most_common(5)
            return ['#%02x%02x%02x' % rgb_tuple[0] for rgb_tuple in most_common]
        except:
            return ["Could not extract"]

    def export_reports(self):
        acc_df = pd.DataFrame(self.results["Accessibility"])
        link_df = pd.DataFrame(self.results["Links"])
        print("\n🎨 BRAND PALETTE:", self.results["Palette"])
        print("\n--- ACCESSIBILITY SUMMARY ---")
        print(acc_df.head())
        print("\n--- LINK HEALTH SUMMARY ---")
        print(link_df.head())

        # Save to CSV for the client
        acc_df.to_csv("accessibility_report.csv", index=False)
        link_df.to_csv("link_health_report.csv", index=False)
        print("\n✅ Reports saved as CSV files.")

# --- EXECUTION ---
my_audit = DigitalAuditor("https://www.python.org")
my_audit.run_full_audit()
my_audit.export_reports()









🚀 Initializing Master Audit for: https://www.python.org
------------------------------

🎨 BRAND PALETTE: ['#ffffff', '#f1f1f1', '#000000', '#f9f9f9', '#eeeeee']

--- ACCESSIBILITY SUMMARY ---
                        Source  Status
0  /static/img/python-logo.png  ✅ Pass

--- LINK HEALTH SUMMARY ---
                            URL         Health
0       https://www.python.org/           ✅ OK
1   https://www.python.org/psf/  ❌ Issue (301)
2       https://docs.python.org  ❌ Issue (302)
3             https://pypi.org/           ✅ OK
4  https://www.python.org/jobs/           ✅ OK

✅ Reports saved as CSV files.
